Parte 1 — Importação e Preparação dos Dados

In [6]:
import sys
print(sys.executable)

C:\Users\yeong\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe


In [8]:
import sys
!{sys.executable} -m pip install pandas openpyxl matplotlib seaborn numpy scipy

Defaulting to user installation because normal site-packages is not writeable
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\yeong\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached matplotlib-3.11.0-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached matplotlib-3.11.0-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl (226 kB)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: C:\Users\yeong\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Carregando arquivo
df = pd.read_excel('..\data\Plano De Saúde Database.xlsx', sheet_name='Dados')

# Checando primeiras linhas
df.head()

<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
C:\Users\yeong\AppData\Local\Temp\ipykernel_9068\3218168652.py:8: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_excel('..\data\Plano De Saúde Database.xlsx', sheet_name='Dados')


,Idade,IMC,Qte_Filhos,Fumante,Região,Custo_Saude,Sexo
0,19,27.900,0,Sim,Centro,1688.492400,Feminino
1,18,33.770,1,Não,Sudeste,172.555230,Masculino
2,28,33.000,3,Não,Sudeste,444.946200,Masculino
3,33,22.705,0,Não,Norte,2198.447061,Masculino
4,32,28.880,0,Não,Norte,386.685520,Masculino


In [16]:
# Criando coluna Fumante_Boolean para melhor utilização nos dados na análise
df['Fumante_Boolean'] = df['Fumante'].map({'Sim': 1, 'Não': 0})

#Checando
df.head()

,Idade,IMC,Qte_Filhos,Fumante,Região,Custo_Saude,Sexo,Fumante_Boolean
0,19,27.900,0,Sim,Centro,1688.492400,Feminino,1
1,18,33.770,1,Não,Sudeste,172.555230,Masculino,0
2,28,33.000,3,Não,Sudeste,444.946200,Masculino,0
3,33,22.705,0,Não,Norte,2198.447061,Masculino,0
4,32,28.880,0,Não,Norte,386.685520,Masculino,0


In [18]:
#Criando coluna IMC Faixas para melhor utilização dos dados na análise
def classificar_imc(imc):
    if imc < 17:
        return 'Muito abaixo do peso'
    elif imc < 18.5:
        return 'Abaixo do peso'
    elif imc < 25:
        return 'Peso normal'
    elif imc < 30:
        return 'Acima do peso'
    elif imc < 35:
        return 'Obesidade I'
    elif imc < 40:
        return 'Obesidade II (Severa)'
    else:
        return 'Obesidade III (Mórbida)'

df['IMC Faixas'] = df['IMC'].apply(classificar_imc)

#Checando
df.head()


,Idade,IMC,Qte_Filhos,Fumante,Região,Custo_Saude,Sexo,Fumante_Boolean,IMC Faixas
0,19,27.900,0,Sim,Centro,1688.492400,Feminino,1,Acima do peso
1,18,33.770,1,Não,Sudeste,172.555230,Masculino,0,Obesidade I
2,28,33.000,3,Não,Sudeste,444.946200,Masculino,0,Obesidade I
3,33,22.705,0,Não,Norte,2198.447061,Masculino,0,Peso normal
4,32,28.880,0,Não,Norte,386.685520,Masculino,0,Acima do peso


Para melhor aproveitar as informações disponibilizadas, era necessário criar duas novas colunas:
- Fumante_Boolean, para que fosse possível contabilizar o número de fumantes e realizar as operações adequadas
- IMC Faixas, para dividir em grupos os funcionários, e assim obter gráficos mais legíveis

A tabela de faixas de IMC utilizada foi a adotada pela OMS, disponível em https://calculacentro.com/calculadora/imc

Parte 2 — Análise Exploratória

Tabelas de Frequência (variáveis categóricas)
    Tabela de frequência para Sexo

In [19]:
#Criando tabela de frequência para a variável Sexo
freq_sexo = df['Sexo'].value_counts().reset_index()
freq_sexo.columns = ['Sexo', 'Freq Absoluta']
freq_sexo['Freq Relativa'] = freq_sexo['Freq Absoluta'] / len(df)
freq_sexo['Freq Acumulada'] = freq_sexo['Freq Relativa'].cumsum()
print(freq_sexo)

        Sexo  Freq Absoluta  Freq Relativa  Freq Acumulada
0  Masculino            676       0.505232        0.505232
1   Feminino            662       0.494768        1.000000


    Tabela de frequência para Fumante

In [20]:
#Criando tabela de frequência para a variável Fumante
freq_fumante = df['Fumante'].value_counts().reset_index()
freq_fumante.columns = ['Fumante', 'Freq Absoluta']
freq_fumante['Freq Relativa'] = freq_fumante['Freq Absoluta'] / len(df)
freq_fumante['Freq Acumulada'] = freq_fumante['Freq Relativa'].cumsum()
print(freq_fumante)

  Fumante  Freq Absoluta  Freq Relativa  Freq Acumulada
0     Não           1064       0.795217        0.795217
1     Sim            274       0.204783        1.000000


    Tabela de frequência para Região

In [21]:
#Criando tabela de frequência para a variável Região
freq_regiao = df['Região'].value_counts().reset_index()
freq_regiao.columns = ['Região', 'Freq Absoluta']
freq_regiao['Freq Relativa'] = freq_regiao['Freq Absoluta'] / len(df)
freq_regiao['Freq Acumulada'] = freq_regiao['Freq Relativa'].cumsum()
print(freq_regiao)

     Região  Freq Absoluta  Freq Relativa  Freq Acumulada
0   Sudeste            364       0.272048        0.272048
1    Centro            325       0.242900        0.514948
2     Norte            325       0.242900        0.757848
3  Nordeste            324       0.242152        1.000000
